# Demostración de Flota Múltiple (Subrutas VRP)

Vamos a aislar la planta de **Celpack** para entender cómo raciona la carga el motor matemático. 
El problema real que tenías es que la demanda real extraída del dataset para Celpack era muy poca (cabía todo en tu camión de 35 Pallets), por lo que el algoritmo apagaba el Camión #2 y Camión #3 automáticamente para ahorrar constes logísticos.

En esta prueba manual, vamos a inundar a Celpack con **80 pallets** falsos. Esto forzará al `LogisticsSolver` a romper las rutas y despachar físicamente los **3 tráilers** simultáneamente.

In [1]:
import polars as pl
import json
from src.engine.solver import LogisticsSolver
from src.utils.geo import GeoUtils

# 1. Cargamos Mengíbar y Celpack
with open('data/locations_smurfit.json', 'r', encoding='utf-8') as f:
    plants_data = json.load(f)

celpack = next(p for p in plants_data['carton_plants'] if p['id'] == 'CP_CELPACK')

# 2. Inyectamos hiper-demanda (4 megaclientes de 20 pallets = 80 Pallets)
clientes_pesados = [
    {'id': 'C1', 'type': 'customer', 'name': 'Cliente Gigante A', 'lat': 39.8, 'lng': -7.6, 'demanda_pallets': 20, 'parent_cp': 'CP_CELPACK'},
    {'id': 'C2', 'type': 'customer', 'name': 'Cliente Gigante B', 'lat': 39.9, 'lng': -7.7, 'demanda_pallets': 20, 'parent_cp': 'CP_CELPACK'},
    {'id': 'C3', 'type': 'customer', 'name': 'Cliente Gigante C', 'lat': 39.7, 'lng': -7.5, 'demanda_pallets': 20, 'parent_cp': 'CP_CELPACK'},
    {'id': 'C4', 'type': 'customer', 'name': 'Cliente Gigante D', 'lat': 39.6, 'lng': -7.8, 'demanda_pallets': 20, 'parent_cp': 'CP_CELPACK'}
]
celpack['customers'] = clientes_pesados

data_aislada = {'paper_plant': plants_data['paper_plant'], 'carton_plants': [celpack]}

print('Escenario montado: Demanda concentrada = 80 pallets. Capacidad max = 35 pallets por tráiler.')

2026-03-22 13:02:12,905 | WARNING  | config | GOOGLE_MAPS_API_KEY no está configurada. Se usará estimación Haversine.


Escenario montado: Demanda concentrada = 80 pallets. Capacidad max = 35 pallets por tráiler.


In [2]:
# 3. Instanciamos el Motor VRP concediéndole 3 camiones a Celpack
geo_engine = GeoUtils(api_type='haversine') # Usamos distancias lineales en RAM para resolver en 1 segundo
solver = LogisticsSolver(data_aislada, geo_engine=geo_engine)

print("Optimizando...")
routes = solver.solve(
    n_clientes=10,
    varias_plantas=False,
    max_pallets_ruta=35,
    max_search_time=15,
    flota_por_planta={'CP_CELPACK': 3} # Forzamos disponibilidad múltiple
)

print(f'Optimización finalizada con éxito. Camiones despachados: {len(routes)}')
print(f'Visita el Log de Descartes del engine en: {solver.drop_log_path}')

2026-03-22 13:02:13,564 | INFO     | src.engine.solver | Nodos parseados: 6 (1 depósito, 1 plantas, 4 clientes)
2026-03-22 13:02:13,565 | INFO     | src.utils.geo | Calculando matriz de distancias con Haversine (línea recta)...
2026-03-22 13:02:13,566 | INFO     | src.engine.solver | Nodos parseados para optimizar (con muelles virtuales): 8
2026-03-22 13:02:13,567 | INFO     | src.engine.solver | Modo: FLOTA ESPECÍFICA (Clonación de Nodos) | max_plantas_ruta=1 | n_clientes=10 | vehículos=3
2026-03-22 13:02:13,579 | INFO     | src.engine.solver | Iniciando optimización (3 vehículos, límite 15s, algoritmo: GUIDED_LOCAL_SEARCH)...
2026-03-22 13:02:13,579 | INFO     | src.engine.solver | Estadísticas de Matriz: Min(>0)=14018.5m, Max=527989.3m


Optimizando...


2026-03-22 13:02:28,581 | INFO     | src.engine.solver | Solver terminó con éxito (status 1).
2026-03-22 13:02:28,581 | INFO     | src.engine.solver | Solución encontrada: 3 rutas activas.
2026-03-22 13:02:28,612 | INFO     | src.engine.drop_logger | DropLogger ha generado el reporte oficial de rechazos en: logs\descartados_motivos.log


Optimización finalizada con éxito. Camiones despachados: 3
Visita el Log de Descartes del engine en: logs\descartados_motivos.log


In [ ]:
# 4. Mostramos el resultado limpio en una Tabla Polars
tabla = []
for i, route in enumerate(routes):
    carga = sum(n.get('demanda_pallets', 0) for n in route if n['type'] == 'customer')
    secuencia = ' ➔ '.join([r['name'] for r in route])
    
    tabla.append({
        'ID Camión': f'Camion #{i+1}',
        'Planta Origen': 'CELPACK',
        'Llenado Carga': f'{carga} / 35 P',
        'Ruta Optimizada (Viaje Real)': secuencia
    })

df = pl.DataFrame(tabla)

ID Camión,Planta Origen,Llenado Carga,Ruta Optimizada (Viaje Real)
str,str,str,str
"""Camion #1""","""CELPACK""","""20 / 35 P""","""Mengíbar ➔ Celpack ➔ Cliente G…"
"""Camion #2""","""CELPACK""","""20 / 35 P""","""Mengíbar ➔ Celpack (Muelle 2) …"
"""Camion #3""","""CELPACK""","""20 / 35 P""","""Mengíbar ➔ Celpack (Muelle 3) …"
